# 12. 子圖設計

學習如何使用子圖來模組化複雜的工作流程。

---

## 🎯 學習目標

- ✅ 建立可重用的子圖
- ✅ 在主圖中嵌入子圖
- ✅ 設計模組化的工作流程

In [1]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

## 12.1 建立子圖

In [2]:
class SubState(TypedDict):
    data: str
    processed: bool

def sub_step1(state): 
    print("    📦 子圖步驟 1")
    return {"data": f"[步驟1] {state['data']}"}

def sub_step2(state): 
    print("    📦 子圖步驟 2")
    return {"data": f"[步驟2] {state['data']}", "processed": True}

subgraph = StateGraph(SubState)
subgraph.add_node("step1", sub_step1)
subgraph.add_node("step2", sub_step2)
subgraph.add_edge(START, "step1")
subgraph.add_edge("step1", "step2")
subgraph.add_edge("step2", END)
sub_app = subgraph.compile()
print("✅ 子圖已建立")

✅ 子圖已建立


In [3]:
result = sub_app.invoke({"data": "測試", "processed": False})
print(f"結果: {result}")

    📦 子圖步驟 1
    📦 子圖步驟 2
結果: {'data': '[步驟2] [步驟1] 測試', 'processed': True}


## 12.2 嵌入主圖

In [4]:
class MainState(TypedDict):
    input: str
    data: str
    processed: bool
    output: str

def prepare(state): 
    print("  🔧 準備")
    return {"data": state['input']}

def finalize(state): 
    print("  🔧 完成")
    return {"output": state['data']}

main = StateGraph(MainState)
main.add_node("prepare", prepare)
main.add_node("process", sub_app)  # 嵌入子圖
main.add_node("finalize", finalize)
main.add_edge(START, "prepare")
main.add_edge("prepare", "process")
main.add_edge("process", "finalize")
main.add_edge("finalize", END)
main_app = main.compile()
print("✅ 主圖已建立")

✅ 主圖已建立


In [5]:
result = main_app.invoke({"input": "資料", "data": "", "processed": False, "output": ""})
print(f"輸出: {result['output']}")

  🔧 準備
    📦 子圖步驟 1
    📦 子圖步驟 2
  🔧 完成
輸出: [步驟2] [步驟1] 資料


## 💡 重點

```python
# 嵌入子圖
main_graph.add_node("name", sub_app)
```

---

下一步：[13. 最佳實踐](13_best_practices.ipynb)